# SpaPath NSCLC workflow

This notebook follows one linear workflow: load the single-cell reference and FOV_3 disease data, preprocess and build graphs, learn initial embeddings, cluster, integrate, detect pathological regions, construct the all-gene disease dataset, and analyze bidirectional cell-cell communication.

## 0. Setup

In [ ]:
from pathlib import Path
import sys
import warnings

import scanpy as sc
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent

SCRIPTS_DIR = PROJECT_ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import spapath_model
import spapath_utils

warnings.filterwarnings("ignore")

In [ ]:
DATASET_ID = "NSCLC"
REFERENCE_SECTION = "scRNA_Sample1"
DISEASE_SECTION = "FOV_3"
DEVICE = "cuda"
SEED = 123

DATA_ROOT = Path("/share/data/wuyd")
PROCESSED_DIR = DATA_ROOT / "SpaPAD" / "processed" / DATASET_ID
OUTPUT_DIR = DATA_ROOT / "SpaPAD" / "outs" / DATASET_ID
FIGURE_DIR = OUTPUT_DIR / "fig"

for output_path in (OUTPUT_DIR, FIGURE_DIR):
    spapath_utils.create_dir(output_path)

adata_type_map = {
    REFERENCE_SECTION: "sc",
    DISEASE_SECTION: "ST",
}
sections = list(adata_type_map)

CELLTYPE_PALETTE = {
    "B-cell": "#006E54",
    "NK": "#00A381",
    "T CD4 memory": "#38B48B",
    "T CD4 naive": "#00A497",
    "T CD8 memory": "#80ABA9",
    "T CD8 naive": "#5C9291",
    "Treg": "#478384",
    "endothelial": "#6E7955",
    "epithelial": "#5A544B",
    "fibroblast": "#F6BFBC",
    "mDC": "#F5B1AA",
    "macrophage": "#F5B199",
    "mast": "#E597B2",
    "monocyte": "#EE827C",
    "neutrophil": "#DEB068",
    "pDC": "#BF794E",
    "plasmablast": "#A59564",
    "tumor": "#96514D",
}

## 1. Read reference and disease data

In [ ]:
reference_adata = sc.read_h5ad(PROCESSED_DIR / f"{REFERENCE_SECTION}.h5ad")
disease_adata = sc.read_h5ad(PROCESSED_DIR / f"{DISEASE_SECTION}.h5ad")

## 2. Preprocess data and build graphs

In [ ]:
batch_list = [reference_adata, disease_adata]
adata_full, disease_adata_all_genes = spapath_utils.preprocess(
    adata_list=batch_list,
    adata_type_map=adata_type_map,
    full_num_hvgs=3000,
    min_genes_qc=10,
    min_cells_qc=10,
)

adata_full = spapath_utils.build_graph_GAT_plus(
    adata_full=adata_full,
    adata_type_map=adata_type_map,
    K=8,
    img_threshold=0.0,
)

## 3. Learn initial embeddings

In [ ]:
model = spapath_model.Model(
    adata_full=adata_full,
    adata_type_map=adata_type_map,
    lr_pre=1e-4,
    lr=1e-4,
    n_pre_training_steps=500,
    n_training_steps=300,
    device=DEVICE,
    seed=SEED,
)

adata_full = model.initial_embedding()

## 4. Cluster observations

In [ ]:
adata_full = model.clustering(
    init_res=1.5,
    intopk=40,
)

## 5. Integrate reference and disease data

In [ ]:
adata_full = model.integrate(topk=40)

## 6. Detect pathological regions

In [ ]:
adata_full = spapath_utils.detection(
    adata=adata_full,
    embed="cell_embed",
    section_ids=sections,
    label_core="Pathological regions",
    label_other="Healthy-like regions",
    core_types=["tumor"],
    celltype_key="CellType",
    batch_key="batch",
    seed=SEED,
    neighbors=30,
    threshold=0.05,
    strategy="individual",
)

display(adata_full.uns["result"])

In [ ]:
detection_figure = spapath_utils.plot_detection_umap(
    adata=adata_full,
    embed="cell_embed",
    section_id=DISEASE_SECTION,
    batch_key="batch",
    label_key="pred_label",
    celltype_key="CellType",
    celltype_palette=CELLTYPE_PALETTE,
    seed=SEED,
    point_size=5,
    save=FIGURE_DIR / f"{DISEASE_SECTION}_detection_umap.png",
)

## 7. Build the all-gene disease dataset

In [ ]:
disease_data = spapath_utils.build_disease_data(
    adata_full=adata_full,
    disease_adata_all_genes=disease_adata_all_genes,
    disease_section=DISEASE_SECTION,
)

## 8. Analyze cell-cell communication

In [ ]:
ccc_summary = spapath_utils.run_ccc_analysis(
    adata=disease_data,
    platform="cosmx",
    sender_label="Pathological regions",
    receiver_label="Healthy-like regions",
    seed=SEED,
)

display(ccc_summary)